# TP01: Python Web Scraping Case-study

**University of Eloued** - Faculty of Exact Sciences - Department of Computer Science
**Master 2: IA & Data Science** - Module: Big Data Analytics - 2026/2027

**Student:** BELLOUM MOHAMMED YACINE

**Target website:** [The Quotations Page](https://www.quotationspage.com/) (educational site, 28,000+ quotes)





## 1. Setup and configuration

In [ ]:
import re
import time
import string
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.quotationspage.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (University TP - educational scraping)"}

TARGET_ROWS = 2000

DELAY = 1
PRINT_HTML = True
HTML_PREVIEW_CHARS = 500
session = requests.Session()
session.headers.update(HEADERS)

## 2. Q1 - Fetch pages and display the HTML source
The function below downloads a page, checks the status code, retries on failure, prints the HTML, and waits `DELAY` seconds.

In [ ]:
def get_html(url, retries=3):
    """Download a page and return its HTML text (None on failure)."""
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=20)
            r.raise_for_status()                 # raises an error if status is not 200
            r.encoding = r.apparent_encoding
            if PRINT_HTML:
                print(f"\n===== {url} (status {r.status_code}) =====")
                print(r.text[:HTML_PREVIEW_CHARS])
                print("...")
            time.sleep(DELAY)
            return r.text
        except requests.RequestException as e:
            print(f"[error] {url} -> {e} (attempt {attempt + 1}/{retries})")
            time.sleep(2)
    return None

In [ ]:
# Quick test on one page
html = get_html(f"{BASE}/quotes/Albert_Einstein/")
print("Page length:", len(html), "characters")


===== https://www.quotationspage.com/quotes/Albert_Einstein/ (status 200) =====
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN">

<html>
<head>
	<title>Albert Einstein Quotes - The Quotations Page</title>
<meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1" >
<meta name="MSSmartTagsPreventParsing" content="TRUE">
<LINK REL="shortcut icon" HREF="/favicon.ico" TYPE="image/x-icon">
<LINK REL="stylesheet" type="text/css" HREF="/qp.css">
<script type="text/javascript">
  var _gaq = _gaq || [];
  _gaq.push(['_setAccount', 'UA-1805249-2']);

...
Page length: 27567 characters


## 3. Q2 - Extract quotes with BeautifulSoup
For every quote we extract: **id, text, author, subjects (tags), quote URL, source page**.

In [ ]:
QUOTE_TITLE = "Click for further information about this quotation"

def parse_quotes(soup, page_url):
    """Extract all quotes from one parsed page."""
    rows = []
    for a in soup.find_all("a", title=QUOTE_TITLE):
        text = a.get_text(strip=True)
        href = urljoin(BASE, a["href"])
        m = re.search(r"/quote/(\d+)\.html", href)
        quote_id = m.group(1) if m else None

        b = a.find_next("b")                       # author name is in the next <b> tag
        author = b.get_text(strip=True) if b else None

        subjects = []                              # subject links until the next quote starts
        for el in a.next_elements:
            if getattr(el, "name", None) == "a":
                if el.get("title") == QUOTE_TITLE:
                    break
                if "/subjects/" in el.get("href", ""):
                    subjects.append(el.get_text(strip=True))

        rows.append({
            "quote_id": quote_id,
            "quote": text,
            "author": author,
            "subjects": ", ".join(subjects),
            "quote_url": href,
            "source_page": page_url,
        })
    return rows


def get_next_page(soup, current_url):
    """Return the URL of the 'Next Page' link, or None if it is the last page."""
    link = soup.find("a", string=re.compile(r"Next Page"))
    return urljoin(current_url, link["href"]) if link else None

In [ ]:
# Test the extraction on the page fetched above
soup = BeautifulSoup(html, "html.parser")
test_rows = parse_quotes(soup, f"{BASE}/quotes/Albert_Einstein/")
print(len(test_rows), "quotes found on this page")
pd.DataFrame(test_rows).head()

20 quotes found on this page


,quote_id,quote,author,subjects,quote_url,source_page
0,248,Before God we are all equally wise - and equal...,Albert Einstein,"God, Equality",https://www.quotationspage.com/quote/248.html,https://www.quotationspage.com/quotes/Albert_E...
1,26184,Do not worry about your difficulties in Mathem...,Albert Einstein,Mathematics,https://www.quotationspage.com/quote/26184.html,https://www.quotationspage.com/quotes/Albert_E...
2,28918,Ethical axioms are found and tested not very d...,Albert Einstein,,https://www.quotationspage.com/quote/28918.html,https://www.quotationspage.com/quotes/Albert_E...
3,38987,Every day I remind myself that my inner and ou...,Albert Einstein,,https://www.quotationspage.com/quote/38987.html,https://www.quotationspage.com/quotes/Albert_E...
4,9049,Few people are capable of expressing with equa...,Albert Einstein,,https://www.quotationspage.com/quote/9049.html,https://www.quotationspage.com/quotes/Albert_E...


## 4. Crawl all pages
1. Collect all author links from the A-Z index.
2. Visit each author page and follow the pagination.
3. Stop when `TARGET_ROWS` is reached.

In [ ]:
def get_author_urls():
    """Collect author page links from the A..Z index pages."""
    author_urls = []
    for letter in string.ascii_uppercase:
        page = get_html(f"{BASE}/quotes/{letter}.html")
        if not page:
            continue
        s = BeautifulSoup(page, "html.parser")
        for a in s.find_all("a", href=True):
            full = urljoin(BASE, a["href"])
            path = full.replace(BASE, "")
            if (path.startswith("/quotes/") and not path.endswith(".html")
                    and path.strip("/") != "quotes" and full not in author_urls):
                author_urls.append(full)
        if len(author_urls) >= 1000:       # enough authors to reach 1000+ quotes
            break
    return author_urls

In [ ]:
all_rows, seen_ids = [], set()

author_urls = get_author_urls()
print(f"\nAuthors collected: {len(author_urls)}")

for author_url in author_urls:
    url = author_url
    while url and len(all_rows) < TARGET_ROWS:
        page = get_html(url)
        if not page:
            break
        soup = BeautifulSoup(page, "html.parser")
        for row in parse_quotes(soup, url):
            if row["quote_id"] not in seen_ids:
                seen_ids.add(row["quote_id"])
                all_rows.append(row)
        print(f"-> total so far: {len(all_rows)} quotes")
        url = get_next_page(soup, url)
    if len(all_rows) >= TARGET_ROWS:
        break

print("\nScraping finished.")

Streaming output truncated to the last 5000 lines.
<meta name="MSSmartTagsPreventParsing" content="TRUE">
<LINK REL="shortcut icon" HREF="/favicon.ico" TYPE="image/x-icon">
<LINK REL="stylesheet" type="text/css" HREF="/qp.css">
<script type="text/javascript">
  var _gaq = _gaq || [];
  _gaq.push(['_setAccount', 'UA-1805249-2'])
...
-> total so far: 1035 quotes

===== https://www.quotationspage.com/quotes/Sir_Arnold_Bax/ (status 200) =====
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN">

<html>
<head>
	<title>Sir Arnold Bax Quotes - The Quotations Page</title>
<meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1" >
<meta name="MSSmartTagsPreventParsing" content="TRUE">
<LINK REL="shortcut icon" HREF="/favicon.ico" TYPE="image/x-icon">
<LINK REL="stylesheet" type="text/css" HREF="/qp.css">
<script type="text/javascript">
  var _gaq = _gaq || [];
  _gaq.push(['_setAccount', 'UA-1805249-2']);
 
...
-> total so far: 1036 quotes

===== https://www.quotationsp

## 5. Q3 and Q4 - Save to CSV and verify 1,000+ rows

In [ ]:
df = pd.DataFrame(all_rows).drop_duplicates(subset="quote").reset_index(drop=True)
df.to_csv("quotes.csv", index=False, encoding="utf-8-sig")

print(f"Rows in file: {len(df)}")
print("OK: requirement satisfied (>= 1000 rows)" if len(df) >= 1000 else "Requirement NOT satisfied")
df.head(10)

Rows in file: 2000
OK: requirement satisfied (>= 1000 rows)


,quote_id,quote,author,subjects,quote_url,source_page
0,33098,Anarchism is founded on the observation that s...,Edward Abbey,,https://www.quotationspage.com/quote/33098.html,https://www.quotationspage.com/quotes/Edward_A...
1,33113,No tyranny is so irksome as petty tyranny: the...,Edward Abbey,,https://www.quotationspage.com/quote/33113.html,https://www.quotationspage.com/quotes/Edward_A...
2,33129,"One man alone can be pretty dumb sometimes, bu...",Edward Abbey,,https://www.quotationspage.com/quote/33129.html,https://www.quotationspage.com/quotes/Edward_A...
3,39968,Sentiment without action is the ruin of the soul.,Edward Abbey,,https://www.quotationspage.com/quote/39968.html,https://www.quotationspage.com/quotes/Edward_A...
4,41688,Whenever I see a photograph of some sportsman ...,Edward Abbey,Hunting,https://www.quotationspage.com/quote/41688.html,https://www.quotationspage.com/quotes/Edward_A...
5,24877,"Abbott: Now, on the St. Louis team we have Who...",Bud Abbott,,https://www.quotationspage.com/quote/24877.html,https://www.quotationspage.com/quotes/Bud_Abbott/
6,39726,If the people who make the decisions are the p...,John Abrams,,https://www.quotationspage.com/quote/39726.html,https://www.quotationspage.com/quotes/John_Abr...
7,38157,The test for whether or not you can hold a job...,Bella Abzug,,https://www.quotationspage.com/quote/38157.html,https://www.quotationspage.com/quotes/Bella_Ab...
8,29546,A man whose life has been dishonourable is not...,Lucius Accius,,https://www.quotationspage.com/quote/29546.html,https://www.quotationspage.com/quotes/Lucius_A...
9,29386,Let them hate so long as they fear.(Oderint Du...,Lucius Accius,,https://www.quotationspage.com/quote/29386.html,https://www.quotationspage.com/quotes/Lucius_A...


In [ ]:
# Data overview
print(df.shape)
print(df.isna().sum())
print("\nTop 10 authors:")
print(df["author"].value_counts().head(10))

(2000, 6)
quote_id       0
quote          0
author         0
subjects       0
quote_url      0
source_page    0
dtype: int64

Top 10 authors:
author
Bible                   232
Aristotle                70
Anonymous                57
Jane Austen              56
Lois McMaster Bujold     46
Gordon Atkinson          45
Woody Allen              42
Sir Francis Bacon        34
Aesop                    33
James M. Barrie          30
Name: count, dtype: int64


In [ ]:
# Download the CSV (Google Colab only)
try:
    from google.colab import files
    files.download("quotes.csv")
except ImportError:
    print("Not on Colab: quotes.csv is saved in the current folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>